## Router Multi-Agent Design Pattern with LangChain

![router_workflow](./Images/router_workflow.png)

### Installing Utilities and Libraries

In [ ]:
%pip install langchain-anthropic==1.5.4 anthropic==0.120.2 python-dotenv==1.2.2

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv
from typing import Literal, TypedDict
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command

load_dotenv()
anthropic_api_key = os.getenv("CLAUDE_API_KEY")
anthropic_model_name = os.getenv("CLAUDE_MODEL_NAME")

### Instantiate the ChatAnthropic Class

In [ ]:
from langchain_anthropic import ChatAnthropic

model = ChatAnthropic(
    model_name = anthropic_model_name,
    api_key = anthropic_api_key
)

### Define the State

In [ ]:
class RouterState(TypedDict):
    query: str
    final_answer: str

### Create Specialized Agents

In [15]:
sales_agent = create_agent(
    model=model,
    tools=[],
    system_prompt=(
        "You are a sales specialist. "
        "Answer questions about products, pricing, "
        "plans, purchases, and upgrades."
    ),
)


support_agent = create_agent(
    model=model,
    tools=[],
    system_prompt=(
        "You are a technical support specialist. "
        "Help users troubleshoot technical problems, "
        "login issues, errors, and product usage."
    ),
)


billing_agent = create_agent(
    model=model,
    tools=[],
    system_prompt=(
        "You are a billing specialist. "
        "Answer questions about invoices, payments, "
        "refunds, and billing issues."
    ),
)

### Create the Router

In [ ]:
def route_query(
    state: RouterState,
) -> Command[
    Literal[
        "sales_agent",
        "support_agent",
        "billing_agent",
    ]
]:

    query = state["query"].lower()

    if any(
        word in query
        for word in [
            "price",
            "pricing",
            "purchase",
            "buy",
            "upgrade",
            "plan",
        ]
    ):
        selected_agent = "sales_agent"

    elif any(
        word in query
        for word in [
            "invoice",
            "payment",
            "refund",
            "charged",
            "billing",
        ]
    ):
        selected_agent = "billing_agent"

    else:
        selected_agent = "support_agent"

    print(f"Router → {selected_agent}")

    return Command(
        goto=selected_agent
    )

### Create the Agent Nodes for Execution

In [ ]:
def call_sales_agent(state: RouterState):

    result = sales_agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content=state["query"]
                )
            ]
        }
    )

    return {
        "final_answer":
            result["messages"][-1].content
    }


def call_support_agent(state: RouterState):

    result = support_agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content=state["query"]
                )
            ]
        }
    )

    return {
        "final_answer":
            result["messages"][-1].content
    }


def call_billing_agent(state: RouterState):

    result = billing_agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content=state["query"]
                )
            ]
        }
    )

    return {
        "final_answer":
            result["messages"][-1].content
    }

### Build the Router Graph

In [ ]:
builder = StateGraph(RouterState)

builder.add_node(
    "router",
    route_query,
)

builder.add_node(
    "sales_agent",
    call_sales_agent,
)

builder.add_node(
    "support_agent",
    call_support_agent,
)

builder.add_node(
    "billing_agent",
    call_billing_agent,
)


builder.add_edge(
    START,
    "router",
)

builder.add_edge(
    "sales_agent",
    END,
)

builder.add_edge(
    "support_agent",
    END,
)

builder.add_edge(
    "billing_agent",
    END,
)


graph = builder.compile()

### Generate the Mermaid Diagram of the Workflow

In [ ]:
png = graph.get_graph().draw_mermaid_png()

with open("router_workflow.png", "wb") as f:
    f.write(png)

### Invoke the Workflow

Queries to try:

1) "I was charged twice for my subscription."

2) "How much does the premium plan cost?"
"""

In [ ]:
result = graph.invoke(
    {
        "query":
             "I was charged twice for my subscription"
    }
)

print("\nFinal Answer:")
print(result["final_answer"])